# FQ-SSIUU Kaggle Gate-0 Environment Audit
This probe observes the raw pre-installed Kaggle environment.

In [ ]:

import os
import platform
import psutil
import time
import uuid
import json
import subprocess
import importlib.metadata
import datetime

RUN_ID = str(uuid.uuid4())
TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).isoformat()
PROBE_VERSION = "1.0.0"
WORKING_DIR = "/kaggle/working/"
SCHEMA_VERSION = "1.0"


In [ ]:

def sanitize_env(env_vars):
    safe = {}
    unsafe_keywords = ["token", "key", "secret", "password", "credential", "auth", "cookie"]
    for k, v in env_vars.items():
        k_lower = k.lower()
        if any(kw in k_lower for kw in unsafe_keywords):
            continue
        safe[k] = v
    return safe

runtime_data = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "python_version": platform.python_version(),
    "python_executable": os.sys.executable,
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "processor": platform.processor(),
    "hostname": platform.node(),
    "cpu_count": psutil.cpu_count(logical=True),
    "total_system_memory": psutil.virtual_memory().total,
    "available_system_memory": psutil.virtual_memory().available,
    "total_disk_space": psutil.disk_usage('/').total,
    "available_disk_space": psutil.disk_usage('/').free,
    "current_working_directory": os.getcwd(),
    "safe_environment_variables": sanitize_env(dict(os.environ))
}


In [ ]:

try:
    nvidia_smi_output = subprocess.check_output(["nvidia-smi"], text=True)
except Exception as e:
    nvidia_smi_output = str(e)

with open(os.path.join(WORKING_DIR, "nvidia_smi.log"), "w") as f:
    f.write(nvidia_smi_output)

gpu_data = {
    "nvidia_smi_output_captured": True
}

try:
    import torch
    gpu_data["cuda_available"] = torch.cuda.is_available()
    gpu_data["cuda_device_count"] = torch.cuda.device_count()
    gpu_data["devices"] = []
    for i in range(torch.cuda.device_count()):
        dev_info = {
            "name": torch.cuda.get_device_name(i),
            "compute_capability": torch.cuda.get_device_capability(i),
            "total_memory": torch.cuda.get_device_properties(i).total_memory,
            "allocated_memory": torch.cuda.memory_allocated(i),
            "reserved_memory": torch.cuda.memory_reserved(i)
        }
        gpu_data["devices"].append(dev_info)
    gpu_data["pytorch_version"] = torch.__version__
    gpu_data["pytorch_cuda_build_version"] = torch.version.cuda
    gpu_data["cudnn_version"] = torch.backends.cudnn.version()
    gpu_data["is_bf16_supported"] = torch.cuda.is_bf16_supported()
    gpu_data["tf32_available"] = torch.backends.cuda.matmul.allow_tf32
    gpu_data["sdpa_exists"] = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
except Exception as e:
    gpu_data["error"] = str(e)

runtime_data["gpu_observations"] = gpu_data

with open(os.path.join(WORKING_DIR, "gate0_environment_manifest.json"), "w") as f:
    json.dump(runtime_data, f, indent=2)


In [ ]:

required_packages = [
    "torch", "torchvision", "torchaudio", "triton", "transformers", "datasets",
    "accelerate", "peft", "trl", "huggingface_hub", "bitsandbytes", "captum",
    "numpy", "scipy", "pandas", "lm-eval", "hydra-core", "omegaconf", "matplotlib",
    "seaborn", "typer", "rich", "tqdm", "jsonlines", "pyyaml", "pytest", "ruff"
]

inventory = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "required_packages": {},
    "full_inventory": {}
}

for pkg in required_packages:
    try:
        ver = importlib.metadata.version(pkg)
        inventory["required_packages"][pkg] = {
            "installed": True,
            "version": ver,
            "error": None
        }
    except importlib.metadata.PackageNotFoundError:
        inventory["required_packages"][pkg] = {
            "installed": False,
            "version": None,
            "error": "PackageNotFoundError"
        }
    except Exception as e:
        inventory["required_packages"][pkg] = {
            "installed": False,
            "version": None,
            "error": type(e).__name__ + ": " + str(e)
        }

try:
    all_dists = list(importlib.metadata.distributions())
    for dist in sorted(all_dists, key=lambda d: (d.metadata['Name'] or '').lower()):
        name = dist.metadata['Name']
        if name:
            inventory["full_inventory"][name.lower()] = dist.version
except Exception as e:
    inventory["full_inventory"]["error"] = str(e)

with open(os.path.join(WORKING_DIR, "gate0_package_inventory.json"), "w") as f:
    json.dump(inventory, f, indent=2)

try:
    pip_freeze_output = subprocess.check_output([os.sys.executable, "-m", "pip", "freeze"], text=True)
except Exception as e:
    pip_freeze_output = str(e)
    
with open(os.path.join(WORKING_DIR, "pip_freeze.log"), "w") as f:
    f.write(pip_freeze_output)


In [ ]:

smoke_results = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "checks": {}
}

def run_check(name, fn):
    try:
        obs = fn()
        smoke_results["checks"][name] = {
            "status": "PASS",
            "observed": obs if obs else {},
            "exception_type": None,
            "exception_message": None
        }
    except Exception as e:
        smoke_results["checks"][name] = {
            "status": "FAIL",
            "observed": {},
            "exception_type": type(e).__name__,
            "exception_message": str(e)
        }

def cpu_tensor_check():
    import torch
    t = torch.ones(5)
    res = t + t
    return {"sum": res.sum().item()}

def cuda_fp16_creation():
    import torch
    t = torch.ones(5, device="cuda", dtype=torch.float16)
    return {"device": str(t.device), "dtype": str(t.dtype)}

def cuda_fp16_matmul():
    import torch
    a = torch.randn(10, 10, device="cuda", dtype=torch.float16)
    b = torch.randn(10, 10, device="cuda", dtype=torch.float16)
    c = torch.matmul(a, b)
    return {"shape": list(c.shape)}

def sdpa_check():
    import torch
    import torch.nn.functional as F
    q = torch.randn(1, 1, 10, 10, device="cuda", dtype=torch.float16)
    k = torch.randn(1, 1, 10, 10, device="cuda", dtype=torch.float16)
    v = torch.randn(1, 1, 10, 10, device="cuda", dtype=torch.float16)
    out = F.scaled_dot_product_attention(q, k, v)
    return {"shape": list(out.shape)}

def nf4_check():
    if not inventory["required_packages"].get("bitsandbytes", {}).get("installed"):
        smoke_results["checks"]["nf4_quantise"] = {
            "status": "SKIP",
            "observed": {},
            "exception_type": None,
            "exception_message": "bitsandbytes not installed"
        }
        return
    import torch
    try:
        import bitsandbytes as bnb
        t = torch.randn(16, 16, device="cuda", dtype=torch.float16)
        out, state = bnb.functional.quantize_4bit(t, quant_type="nf4")
        deq = bnb.functional.dequantize_4bit(out, state, quant_type="nf4")
        return {"quantized_shape": list(out.shape), "dequantized_shape": list(deq.shape)}
    except Exception as e:
        raise e

run_check("cpu_tensor", cpu_tensor_check)
run_check("cuda_fp16_creation", cuda_fp16_creation)
run_check("cuda_fp16_matmul", cuda_fp16_matmul)
run_check("sdpa", sdpa_check)

try:
    nf4_check_res = nf4_check()
    if nf4_check_res is not None:
        smoke_results["checks"]["nf4_quantise"] = {
            "status": "PASS",
            "observed": nf4_check_res,
            "exception_type": None,
            "exception_message": None
        }
except Exception as e:
    smoke_results["checks"]["nf4_quantise"] = {
        "status": "FAIL",
        "observed": {},
        "exception_type": type(e).__name__,
        "exception_message": str(e)
    }

try:
    import torch
    peak = torch.cuda.max_memory_allocated()
    smoke_results["checks"]["peak_memory"] = {
        "status": "PASS",
        "observed": {"peak_allocated_bytes": peak},
        "exception_type": None,
        "exception_message": None
    }
except Exception as e:
    pass

try:
    import torch
    torch.cuda.empty_cache()
except:
    pass

with open(os.path.join(WORKING_DIR, "gate0_smoke_tests.json"), "w") as f:
    json.dump(smoke_results, f, indent=2)


In [ ]:

kaggle_execution_status = "PASS" # since it reached here
t4_hardware_status = "FAIL"
cuda_smoke_status = "FAIL"
sdpa_smoke_status = "FAIL"
nf4_smoke_status = "FAIL"
dependency_observation_status = "PARTIAL"

try:
    for dev in gpu_data.get("devices", []):
        if "T4" in dev.get("name", ""):
            t4_hardware_status = "PASS"
            break
except:
    pass

c_create = smoke_results["checks"].get("cuda_fp16_creation", {}).get("status") == "PASS"
c_matmul = smoke_results["checks"].get("cuda_fp16_matmul", {}).get("status") == "PASS"
if c_create and c_matmul:
    cuda_smoke_status = "PASS"

if smoke_results["checks"].get("sdpa", {}).get("status") == "PASS":
    sdpa_smoke_status = "PASS"

nf4_stat = smoke_results["checks"].get("nf4_quantise", {}).get("status")
if nf4_stat == "PASS":
    nf4_smoke_status = "PASS"
elif nf4_stat == "SKIP":
    nf4_smoke_status = "SKIP"

all_required = True
for pkg, info in inventory["required_packages"].items():
    if not info.get("installed"):
        all_required = False
        break
if all_required:
    dependency_observation_status = "PASS"

if t4_hardware_status == "PASS" and cuda_smoke_status == "PASS" and sdpa_smoke_status == "PASS" and nf4_smoke_status == "PASS" and dependency_observation_status == "PASS":
    overall_gate0_status = "PASS"
elif t4_hardware_status == "PASS" and cuda_smoke_status == "PASS" and sdpa_smoke_status == "PASS":
    overall_gate0_status = "PARTIAL"
else:
    overall_gate0_status = "FAIL"

summary = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "statuses": {
        "kaggle_execution_status": kaggle_execution_status,
        "t4_hardware_status": t4_hardware_status,
        "cuda_smoke_status": cuda_smoke_status,
        "sdpa_smoke_status": sdpa_smoke_status,
        "nf4_smoke_status": nf4_smoke_status,
        "dependency_observation_status": dependency_observation_status,
        "overall_gate0_status": overall_gate0_status
    }
}

with open(os.path.join(WORKING_DIR, "gate0_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
